# W&B Sweeps — Fine-tune the Registered Model

Sweeps are W&B's hyperparameter search. A **sweep controller** samples
hyperparameters from a config, hands them to an **agent** that runs
one training trial per proposal, and updates its beliefs from the
metric each trial logs back. The Bayesian method used here proposes
new points based on which regions of the space look most promising —
it doesn't sweep exhaustively, so it converges much faster than grid
search on modest trial counts.

This notebook demonstrates the full loop:

1. **Pull the champion checkpoint** from the Registry (produced by
   `experiment_tracking.ipynb`) — this is the warm-start.
2. **Fine-tune** it under different hyperparameter proposals via a
   Bayesian sweep.
3. **Log every trial checkpoint** as a model artifact candidate.
4. **Promote the sweep winner** back to the Registry under a new
   Collection (`model_hyperparameter_tuned`) so downstream consumers
   can pick it up.

**Prerequisites** — both must be run first:
- `registry.ipynb` — produces `wandb-registry-airfoil_surrogates/datasets:production`.
- `experiment_tracking.ipynb` — produces
  `wandb-registry-airfoil_surrogates/checkpoints:production`.


## 1. Setup

Third-party imports and the section-local constants — no
`physicsx_demo` imports, notebook stands alone.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "physicsx_demo").exists())
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import json
import math
import tempfile
from pathlib import Path

import torch
import torch.nn.functional as F
import wandb

from physicsx_demo.utils.data import cases_from_payload, split_cases, stack_cases
from physicsx_demo.utils.env import load_local_env, resolve_wandb_target
from physicsx_demo.utils.metrics import proxy_metrics, relative_l2, selection_score, slice_regularization
from physicsx_demo.utils.model import TinyTransolver, count_params

load_local_env()

device = torch.device("cpu")
ENTITY, PROJECT = resolve_wandb_target()

REGISTRY_NAME = "airfoil_surrogates"
DATASET_COLLECTION = "datasets"
BASE_COLLECTION = "checkpoints"
TUNED_COLLECTION = "model_hyperparameter_tuned"

DATASET_REGISTRY_PATH = f"wandb-registry-{REGISTRY_NAME}/{DATASET_COLLECTION}:production"
CHECKPOINT_REGISTRY_PATH = f"wandb-registry-{REGISTRY_NAME}/{BASE_COLLECTION}:production"

print(f"torch  = {torch.__version__}")
print(f"wandb  = {wandb.__version__}")
print(f"target = {ENTITY}/{PROJECT}")
print(f"device = {device}")


## 2. Model — same TinyTransolver

The sweep fine-tunes the same architecture the training notebook
produced. The model class lives in `src/physicsx_demo/utils/model.py`,
so this notebook imports it instead of repeating the architecture
definition. A checkpoint `state_dict` still needs the matching
`nn.Module` class before weights can be loaded.

Physics-Attention reference: `docs/explainers/pxtransolver.md`.


In [ ]:
# TinyTransolver, PhysicsAttention, and count_params are imported from
# `physicsx_demo.utils.model`, matching the experiment-tracking notebook.


## 3. Pull the pretrained weights + dataset from W&B

A short `sweep-setup` run consumes both the champion checkpoint and
the synthetic dataset. Using `run.use_artifact(...)` records both as
inputs of this run, so the W&B lineage graph shows the setup step
sourcing from the Registry entry produced by `experiment_tracking.ipynb`.

The checkpoint dict written by that notebook contains
`{state_dict, config, normalization, epoch, selection_score, ...}` — enough to
reinstantiate the global-context `TinyTransolver`, load the weights, and understand the
training-time scaling. We compute
train-only normalisation stats up front so every sweep trial shares
identical inputs.


In [ ]:
# ---- Download both artifacts inside a short setup run ----
with wandb.init(
    entity=ENTITY,
    project=PROJECT,
    job_type="sweep-setup",
    name="sweep-setup",
    tags=["notebook", "sweep", "setup"],
) as setup_run:
    ckpt_art = setup_run.use_artifact(CHECKPOINT_REGISTRY_PATH)
    ckpt_dir = Path(ckpt_art.download())
    ds_art = setup_run.use_artifact(DATASET_REGISTRY_PATH)
    ds_dir = Path(ds_art.download())

    ckpt_file = next(ckpt_dir.glob("*.pt"))
    ckpt = torch.load(ckpt_file, map_location="cpu")

    MODEL_KEYS = ("in_dim", "out_dim", "hidden", "n_layers", "n_slices", "n_heads", "ffn_expansion", "dropout", "use_global_context")
    PRETRAINED_CONFIG = {k: ckpt["config"][k] for k in MODEL_KEYS if k in ckpt["config"]}
    PRETRAINED_STATE_DICT = ckpt["state_dict"]

    print(f"loaded champion checkpoint (pretrain val/pressure_rmse = {ckpt.get('val_pressure_rmse'):.4f}, selection_score = {ckpt.get('selection_score', float('nan')):.4f})")
    print(f"model config: {PRETRAINED_CONFIG}")

    ds_file = next(ds_dir.glob("*.json"))
    with open(ds_file) as f:
        payload = json.load(f)

    cases = cases_from_payload(payload)
    train_cases, val_cases = split_cases(cases, val_fraction=0.2)
    X_train, Y_train = stack_cases(train_cases)
    X_val, Y_val = stack_cases(val_cases)

    FEAT_MEAN = X_train.mean(dim=(0, 1), keepdim=True)
    FEAT_STD = X_train.std(dim=(0, 1), keepdim=True).clamp(min=1e-6)
    TGT_MEAN = Y_train.mean(dim=(0, 1), keepdim=True)
    TGT_STD = Y_train.std(dim=(0, 1), keepdim=True).clamp(min=1e-6)

    X_TRAIN_N = ((X_train - FEAT_MEAN) / FEAT_STD).to(device)
    Y_TRAIN_N = ((Y_train - TGT_MEAN) / TGT_STD).to(device)
    X_VAL_N = ((X_val - FEAT_MEAN) / FEAT_STD).to(device)
    Y_VAL = Y_val.to(device)
    Y_VAL_N = ((Y_val - TGT_MEAN) / TGT_STD).to(device)

    print(f"train: X={tuple(X_TRAIN_N.shape)} Y={tuple(Y_TRAIN_N.shape)}    "
          f"val: X={tuple(X_VAL_N.shape)} Y={tuple(Y_VAL_N.shape)}")

    _baseline = TinyTransolver(**PRETRAINED_CONFIG).to(device)
    _baseline.load_state_dict(PRETRAINED_STATE_DICT)
    _baseline.eval()
    with torch.no_grad():
        _pred_n = _baseline(X_VAL_N)
        _err = _pred_n - Y_VAL_N
        baseline_p_rmse = _err[..., 0].pow(2).mean().sqrt().item()
        _pred = _pred_n * TGT_STD.to(device) + TGT_MEAN.to(device)
        baseline_field_rel_l2 = relative_l2(_pred, Y_VAL).item()
        baseline_proxy = proxy_metrics(val_cases, _pred.cpu().numpy())
        baseline_selection_score = selection_score(
            baseline_p_rmse,
            baseline_field_rel_l2,
            baseline_proxy["val/lift_proxy_mae"],
            baseline_proxy["val/drag_proxy_mae"],
        )

    print(f"warm-start baseline val/pressure_rmse = {baseline_p_rmse:.4f}")
    print(f"warm-start baseline val/selection_score = {baseline_selection_score:.4f}   (sweep goal: beat this)")


## 4. Sweep configuration and training function

Five hyperparameters, all training-time — architecture is fixed by
the loaded `state_dict`:

- **`lr`** — dominant fine-tune knob. Log-uniform `[1e-5, 3e-3]`.
- **`weight_decay`** — regularisation on top of already-trained
  weights. Log-uniform `[1e-6, 1e-3]`.
- **`lambda_slice_usage`** — keeps all slices alive. Uniform `[0.0, 0.05]`.
- **`lambda_slice_sharpness`** — pushes points to commit to fewer slices. Uniform `[0.0, 0.08]`.
- **`batch_size`** — discrete `[4, 8, 16]`. Matters for gradient
  noise on the small 64-case train split.

The `train_fn` closure captures the pretrained state and normalised
tensors from §3, so each trial starts from an identical warm-start
without re-downloading.


In [ ]:
FINE_TUNE_EPOCHS = 5

SWEEP_CONFIG = {
    "method": "bayes",
    "metric": {"name": "val/selection_score", "goal": "minimize"},
    "parameters": {
        "lr":                     {"distribution": "log_uniform_values", "min": 1e-5, "max": 3e-3},
        "weight_decay":           {"distribution": "log_uniform_values", "min": 1e-6, "max": 1e-3},
        "lambda_slice_usage":     {"distribution": "uniform",             "min": 0.0,  "max": 0.05},
        "lambda_slice_sharpness": {"distribution": "uniform",             "min": 0.0,  "max": 0.08},
        "batch_size":             {"values": [4, 8, 16]},
    },
}



def train_fn():
    with wandb.init(
        job_type="sweep-trial",
        tags=["notebook", "sweep", "fine-tune"],
    ) as run:
        cfg = run.config

        # Record source artifacts on this trial too so the lineage graph
        # includes every trial as a consumer of the champion + dataset.
        run.use_artifact(CHECKPOINT_REGISTRY_PATH)
        run.use_artifact(DATASET_REGISTRY_PATH)

        model = TinyTransolver(**PRETRAINED_CONFIG).to(device)
        model.load_state_dict(PRETRAINED_STATE_DICT)
        optim = torch.optim.AdamW(
            model.parameters(),
            lr=cfg.lr,
            weight_decay=cfg.weight_decay,
        )

        n = X_TRAIN_N.size(0)
        val_p_rmse = float("nan")
        val_selection_score = float("nan")
        for epoch in range(FINE_TUNE_EPOCHS):
            # --- train ---
            model.train()
            perm = torch.randperm(n)
            for start in range(0, n, cfg.batch_size):
                idx = perm[start:start + cfg.batch_size]
                xb, yb = X_TRAIN_N[idx], Y_TRAIN_N[idx]
                pred, all_w = model(xb, return_slice_weights=True)
                mse = F.mse_loss(pred, yb)
                aux, _ = slice_regularization(
                    all_w,
                    usage_weight=cfg.lambda_slice_usage,
                    sharpness_weight=cfg.lambda_slice_sharpness,
                )

                loss = mse + aux
                optim.zero_grad()
                loss.backward()
                optim.step()

                run.log({
                    "train/loss": loss.item(),
                    "train/mse": mse.item(),
                    "train/aux": aux.item(),
                })

            # --- validate ---
            model.eval()
            with torch.no_grad():
                pred_v = model(X_VAL_N)
                err = pred_v - Y_VAL_N
                val_p_rmse = err[..., 0].pow(2).mean().sqrt().item()
                val_u_rmse = err[..., 1].pow(2).mean().sqrt().item()
                val_v_rmse = err[..., 2].pow(2).mean().sqrt().item()
                val_vel_rmse = float(((val_u_rmse ** 2 + val_v_rmse ** 2) / 2) ** 0.5)
                pred_v_denorm = pred_v * TGT_STD.to(device) + TGT_MEAN.to(device)
                val_field_rel_l2 = relative_l2(pred_v_denorm, Y_VAL).item()
                val_proxy = proxy_metrics(val_cases, pred_v_denorm.cpu().numpy())
                val_selection_score = selection_score(
                    val_p_rmse,
                    val_field_rel_l2,
                    val_proxy["val/lift_proxy_mae"],
                    val_proxy["val/drag_proxy_mae"],
                )

            run.log({
                "epoch": epoch,
                "val/pressure_rmse": val_p_rmse,
                "val/velocity_rmse": val_vel_rmse,
                "val/u_rmse": val_u_rmse,
                "val/v_rmse": val_v_rmse,
                "val/field_relative_l2": val_field_rel_l2,
                "val/selection_score": val_selection_score,
                **val_proxy,
            })

        # Save this trial's fine-tuned checkpoint as a model artifact.
        # The promotion cell below links the best one into W&B Models Registry.
        tmp = Path(tempfile.mkdtemp())
        ckpt_path = tmp / "airfoil_surrogate_tuned.pt"
        torch.save(
            {
                "state_dict": model.state_dict(),
                "config": PRETRAINED_CONFIG,
                "final_val_pressure_rmse": val_p_rmse,
                "final_val_selection_score": val_selection_score,
                "sweep_hyperparameters": dict(cfg),
                "source_checkpoint_registry_path": CHECKPOINT_REGISTRY_PATH,
                "dataset_registry_path": DATASET_REGISTRY_PATH,
                "fine_tune_epochs": FINE_TUNE_EPOCHS,
            },
            ckpt_path,
        )
        art = wandb.Artifact(
            name="airfoil_surrogate_ckpt_tuned",
            type="model",
            metadata={
                "val_pressure_rmse": val_p_rmse,
                "val_selection_score": val_selection_score,
                "source_checkpoint_registry_path": CHECKPOINT_REGISTRY_PATH,
                "dataset_registry_path": DATASET_REGISTRY_PATH,
                "fine_tune_epochs": FINE_TUNE_EPOCHS,
                "n_params": count_params(model),
                **dict(cfg),
            },
        )
        art.add_file(str(ckpt_path))
        logged = run.log_artifact(art, aliases=["candidate", f"run-{run.id}"])
        logged.wait()

        run.summary["model_checkpoint/artifact"] = logged.qualified_name
        run.summary["model_checkpoint/aliases"] = ["candidate", f"run-{run.id}"]
        run.summary["model_checkpoint/promotable_to_registry"] = True


## 5. Launch the sweep

`wandb.sweep` registers the config with the controller and returns a
`sweep_id`. `wandb.agent` polls the controller for HP proposals and
runs `train_fn` once per proposal, up to `count` times. The controller
uses the composite `val/selection_score` each trial logs to update its beliefs.

Open the printed URL in a browser to watch the sweep populate live.
Each trial logs an `airfoil_surrogate_ckpt_tuned` model artifact with
aliases `candidate` and `run-<id>`, which gives the promotion cell a
concrete checkpoint to link into Models Registry.


In [ ]:
sweep_id = wandb.sweep(SWEEP_CONFIG, entity=ENTITY, project=PROJECT)

print(f"sweep id  = {sweep_id}")
print(f"sweep url = https://wandb.ai/{ENTITY}/{PROJECT}/sweeps/{sweep_id}")

wandb.agent(sweep_id, function=train_fn, count=8)


## 6. Promote the winner into a new Registry Collection

The best trial is the one with the lowest final `val/selection_score`.
We resolve it via `wandb.Api().sweep(...)`, grab its logged checkpoint
artifact, log a small top-trials `wandb.Table`, and `run.link_artifact`
the winning checkpoint into the new Collection
`model_hyperparameter_tuned` under the existing `airfoil_surrogates`
Registry — no `create_registry` needed; Collections are created
implicitly on first link.

This mirrors the production pattern: sweep runs create candidate model
artifacts; only the selected candidate is linked/promoted into the
Registry aliases that downstream systems consume.


In [ ]:
with wandb.init(
    entity=ENTITY,
    project=PROJECT,
    job_type="sweep-promotion",
    name="sweep-winner-promotion",
    tags=["notebook", "sweep", "promotion"],
) as promo_run:
    
    api = wandb.Api(overrides={"entity": ENTITY, "project": PROJECT})
    sweep = api.sweep(f"{ENTITY}/{PROJECT}/{sweep_id}")

    def trial_selection_score(run):
        value = run.summary.get("val/selection_score", float("inf"))
        return float(value) if value is not None else float("inf")

    viable_runs = [r for r in sweep.runs if math.isfinite(trial_selection_score(r))]
    if not viable_runs:
        raise RuntimeError("No completed sweep runs with finite val/selection_score were found.")

    # Best trial by final composite selection score (min).
    best = min(viable_runs, key=trial_selection_score)
    
    best_score = best.summary.get("val/selection_score")
    best_rmse = best.summary.get("val/pressure_rmse")
    print(
        f"winner: {best.name}   val/selection_score = {best_score:.4f}   "
        f"val/pressure_rmse = {best_rmse:.4f}   (warm-start score was {baseline_selection_score:.4f})"
    )

    # Grab the model checkpoint artifact this trial logged.
    tuned_artifacts = [
        artifact for artifact in best.logged_artifacts()
        if artifact.type == "model" and artifact.name.startswith("airfoil_surrogate_ckpt_tuned")
    ]
    if not tuned_artifacts:
        raise RuntimeError(f"Best run {best.name} did not log airfoil_surrogate_ckpt_tuned.")
    best_art = tuned_artifacts[-1]
    best_art.wait()

    top_rows = []
    for rank, trial in enumerate(sorted(viable_runs, key=trial_selection_score), start=1):
        top_rows.append([
            rank,
            trial.name,
            trial.id,
            trial_selection_score(trial),
            trial.summary.get("val/pressure_rmse"),
            trial.config.get("lr"),
            trial.config.get("weight_decay"),
            trial.config.get("lambda_slice_usage"),
            trial.config.get("lambda_slice_sharpness"),
            trial.config.get("batch_size"),
        ])
    promo_run.log({
        "sweep/top_trials": wandb.Table(
            columns=[
                "rank",
                "run_name",
                "run_id",
                "val_selection_score",
                "val_pressure_rmse",
                "lr",
                "weight_decay",
                "lambda_slice_usage",
                "lambda_slice_sharpness",
                "batch_size",
            ],
            data=top_rows,
        )
    })

    target_path = f"wandb-registry-{REGISTRY_NAME}/{TUNED_COLLECTION}"
    registry_aliases = ["production", "sweep-winner", f"sweep-{sweep_id}", f"run-{best.id}"]
    promo_run.use_artifact(best_art)
    promo_run.link_artifact(
        best_art,
        target_path=target_path,
        aliases=registry_aliases,
    )
    promo_run.summary["sweep/winner_run"] = best.name
    promo_run.summary["sweep/winner_run_id"] = best.id
    promo_run.summary["sweep/winner_selection_score"] = best_score
    promo_run.summary["sweep/winner_pressure_rmse"] = best_rmse
    promo_run.summary["sweep/warm_start_selection_score"] = baseline_selection_score
    promo_run.summary["sweep/warm_start_pressure_rmse"] = baseline_p_rmse
    promo_run.summary["sweep/selection_score_delta"] = best_score - baseline_selection_score
    promo_run.summary["sweep/pressure_rmse_delta"] = best_rmse - baseline_p_rmse
    promo_run.summary["model_registry/path"] = target_path
    promo_run.summary["model_registry/artifact"] = best_art.qualified_name
    promo_run.summary["model_registry/aliases"] = registry_aliases
    
    print(f"linked {best_art.name}:{best_art.version} -> {target_path}")
    print(f"aliases: {', '.join(registry_aliases)}")


## 7. Wrap up — what to look at in the W&B UI

**Sweep page** (URL printed in §5)
- Parallel-coordinates plot with all five HPs and the final metric.
- Parameter-importance panel — expect `lr` to dominate.
- 8 individual trial runs, each with train/val curves and a checkpoint
  artifact candidate.

**Lineage graph** (Artifacts tab)
- Every sweep trial run consumes both Registry entries:
  `checkpoints:production` and `datasets:production`. Setup and
  promotion runs consume the same two artifacts.

**Registry**
- Registry `airfoil_surrogates` now has the pipeline Collections:
  - `datasets` — synthetic NACA cases from `registry.ipynb`.
  - `checkpoints` — base model versions from
    `experiment_tracking.ipynb`.
  - `model_hyperparameter_tuned` — the sweep winner checkpoint, aliased
    `production`, `sweep-winner`, `sweep-<id>`, and `run-<id>`.

Downstream notebooks / consumers can pull the tuned model with
`run.use_artifact("wandb-registry-airfoil_surrogates/model_hyperparameter_tuned:production")`.
